# Parameters

In [ ]:
MODULES_FOLDER = "../"

REFERENCE_FOLDER = 'mimic_data/complete_data/'

MICE_DATA_FOLDER = "mice-col/"
PMM_DATA_FOLDER = "pmm/"
MIXED_DATA_FOLDER = "mixed/"

REFERENCE_DATA_COVID = 'patient_covid.csv'
REFERENCE_DATA_CONTROL_PRE = 'patient_control_pre.csv'

COVID_TRAIN_DATA_FILE = 'patient_covid_train.csv'
COVID_TEST_DATA_FILE = 'patient_covid_test.csv'

CONTROL_PRE_TRAIN_DATA_FILE = 'patient_control_pre_train.csv'
CONTROL_PRE_TEST_DATA_FILE = 'patient_control_pre_test.csv'


# Import Data

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import sys

sys.path.append(os.path.abspath(MODULES_FOLDER))

import Modules.DataAnalysisModule as da

# Set the path to the datasets
current_dir = os.getcwd()
folderpath = os.path.join(current_dir, REFERENCE_FOLDER)

df_covid_raw = pd.read_csv(os.path.join(folderpath, REFERENCE_DATA_COVID))
df_control_pre_raw = pd.read_csv(os.path.join(folderpath, REFERENCE_DATA_CONTROL_PRE))

folderpath = os.path.join(current_dir, MICE_DATA_FOLDER)

df_covid_train = pd.read_csv(os.path.join(folderpath, COVID_TRAIN_DATA_FILE))
df_covid_test = pd.read_csv(os.path.join(folderpath, COVID_TEST_DATA_FILE))
data_covid = pd.concat([df_covid_train, df_covid_test], ignore_index=True)

df_control_pre_train = pd.read_csv(os.path.join(folderpath, CONTROL_PRE_TRAIN_DATA_FILE))
df_control_pre_test = pd.read_csv(os.path.join(folderpath, CONTROL_PRE_TEST_DATA_FILE))
data_control_pre = pd.concat([df_control_pre_train, df_control_pre_test], ignore_index=True)

folderpath = os.path.join(current_dir, PMM_DATA_FOLDER)

df_covid_train_pmm = pd.read_csv(os.path.join(folderpath, COVID_TRAIN_DATA_FILE))
df_covid_test_pmm = pd.read_csv(os.path.join(folderpath, COVID_TEST_DATA_FILE))
data_covid_pmm = pd.concat([df_covid_train_pmm, df_covid_test_pmm], ignore_index=True)

df_control_pre_train_pmm = pd.read_csv(os.path.join(folderpath, CONTROL_PRE_TRAIN_DATA_FILE))
df_control_pre_test_pmm = pd.read_csv(os.path.join(folderpath, CONTROL_PRE_TEST_DATA_FILE))
data_control_pre_pmm = pd.concat([df_control_pre_train_pmm, df_control_pre_test_pmm], ignore_index=True)

folderpath = os.path.join(current_dir, MIXED_DATA_FOLDER)

df_covid_train_mixed = pd.read_csv(os.path.join(folderpath, COVID_TRAIN_DATA_FILE))
df_covid_test_mixed = pd.read_csv(os.path.join(folderpath, COVID_TEST_DATA_FILE))
data_covid_mixed = pd.concat([df_covid_train_mixed, df_covid_test_mixed], ignore_index=True)

df_control_pre_train_mixed = pd.read_csv(os.path.join(folderpath, CONTROL_PRE_TRAIN_DATA_FILE))
df_control_pre_test_mixed = pd.read_csv(os.path.join(folderpath, CONTROL_PRE_TEST_DATA_FILE))
data_control_pre_mixed = pd.concat([df_control_pre_train_mixed, df_control_pre_test_mixed], ignore_index=True)

In [ ]:
import re

def process_feature_name(feature):
    match = re.match(r"([^_]+)_(Hematology|Chemistry)_([^_]+)_([^_]+)", feature)
    if match:
        return f"{match.group(1)} ({match.group(4)})"
    return feature

In [ ]:
process_feature_name("pH_Hematology_Urine_units")

In [ ]:
from Modules.config import NUMERICAL, CATEGORICAL

# df_control_pre_raw
# df_covid_raw

aux = df_covid_raw.copy()
aux.rename(columns=process_feature_name, inplace=True)
aux.rename(columns={**NUMERICAL, **CATEGORICAL}, inplace=True)
aux.info()

In [ ]:
x = [x for x in aux.count().values if x != 3355]
valor = sum(x) / len(x) / 3355 * 100
print(f"Valor médio de preenchimento: {valor:.2f}")

In [ ]:
def unit_finder(feature):
    # Procura por: qualquer texto + espaço opcional + (conteúdo dentro de parênteses) no final da string
    # (.+?) -> Grupo 1: O nome do atributo (non-greedy)
    # \s* -> Espaço opcional
    # \(([^()]+)\)$ -> Grupo 2: O que estiver dentro dos parênteses ao final da string
    match = re.search(r"(.+?)\s*\(([^()]+)\)$", feature)
    
    if match:
        atributo = match.group(1).strip()
        unidade = match.group(2).strip()
        
        # Limpeza para casos de 'nan' ou strings vazias
        if unidade.lower() == "nan" or unidade == "":
            unidade = "-"
            
        return (atributo, unidade)
    
    # Se não houver parênteses, o atributo é o nome completo e a unidade é "-"
    return (feature, "-")

In [ ]:
colunas = [unit_finder(x) for x in aux.columns]
colunas = [(x[0],x[1],y) for x,y in zip(colunas, aux.count().values)]
top = 24
colunas = colunas[:top] + sorted(colunas[top:], key=lambda x: x[0].lower())
colunas

## Missing Analysis

In [ ]:
analysis = da.Analysis(df_covid_raw)
analysis.analyze_missing_data(show_info=False)
plots = analysis.get_plot_types()

In [ ]:
# df_control_pre_raw
# df_covid_raw

number = 4

analysis.plot_missing_data(
    plot_type=plots[number],
    top_missing=-1,
    add_title="na coorte COVID-19 (Admissão Primária)",
    figsize=(12, 12),
)

In [ ]:
asdasda

## Compare Distributions

In [ ]:
def compare_structure(df_orig, df_imp):
    """
    Compara Original (sem NaNs) vs Imputado (Completo)
    """
    plt.figure(figsize=(16, 10))
    corr_diff = abs(df_orig.corr() - df_imp.corr())
    sns.heatmap(corr_diff, annot=False, cmap="Reds", fmt=".3f")
    plt.title("Diferença Absoluta nas Matrizes de Correlação")
    plt.show()

In [ ]:
from typing import List, Tuple

def compare_structure_multiple(
    df_pairs: List[Tuple[pd.DataFrame, pd.DataFrame, str]],
    figsize: Tuple[int, int] = (24, 24),
    vmin: float = None,
    vmax: float = None,
    cmap: str = "Reds",
    annot: bool = False,
    cbar: bool = False,  # Alterado para False por padrão
    show_axis_labels: bool = False,  # Novo parâmetro
    show_tick_labels: bool = False  # Novo parâmetro
):
    """
    Compara múltiplos pares de dataframes (Original vs Imputado) em um grid.
    
    Parameters
    ----------
    df_pairs : List[Tuple[pd.DataFrame, pd.DataFrame, str]]
        Lista de tuplas contendo (df_original, df_imputado, label)
        Suporta até 9 comparações (grid 3x3)
    figsize : Tuple[int, int], default=(24, 24)
        Tamanho da figura completa
    vmin : float, optional
        Valor mínimo para escala de cores (None = automático)
    vmax : float, optional
        Valor máximo para escala de cores (None = automático)
    cmap : str, default="Reds"
        Mapa de cores para os heatmaps
    annot : bool, default=False
        Se True, anota valores nas células
    cbar : bool, default=False
        Se True, mostra barra de cores
    show_axis_labels : bool, default=False
        Se True, mostra labels dos eixos (xlabel, ylabel)
    show_tick_labels : bool, default=False
        Se True, mostra os nomes das features nos ticks
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        Figura com todos os subplots
    axes : np.ndarray
        Array de eixos dos subplots
        
    Examples
    --------
    >>> df_pairs = [
    ...     (df_orig_knn, df_imp_knn, "KNN Imputation"),
    ...     (df_orig_mice, df_imp_mice, "MICE Imputation"),
    ...     (df_orig_median, df_imp_median, "Median Imputation")
    ... ]
    >>> fig, axes = compare_structure(df_pairs, cbar=False, show_tick_labels=False)
    >>> plt.tight_layout()
    >>> plt.show()
    """
    
    n_comparisons = len(df_pairs)
    
    if n_comparisons == 0:
        raise ValueError("A lista df_pairs está vazia!")
    
    if n_comparisons > 9:
        raise ValueError(f"Máximo de 9 comparações suportadas, recebido {n_comparisons}")
    
    # Calcular dimensões do grid (3x3 máximo)
    n_rows = int(np.ceil(np.sqrt(n_comparisons)))
    n_cols = int(np.ceil(n_comparisons / n_rows))
    
    # Criar figura e subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    
    # Garantir que axes seja sempre um array 2D
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1 or n_cols == 1:
        axes = axes.reshape(n_rows, n_cols)
    
    # Calcular escala global se não especificada
    if vmin is None or vmax is None:
        all_diffs = []
        for df_orig, df_imp, _ in df_pairs:
            # Remover NaNs do original para comparação justa
            df_orig_clean = df_orig.dropna(axis=1, how='all').dropna(axis=0, how='all')
            
            # Alinhar colunas
            common_cols = df_orig_clean.columns.intersection(df_imp.columns)
            corr_diff = abs(
                df_orig_clean[common_cols].corr() - df_imp[common_cols].corr()
            )
            all_diffs.append(corr_diff.values.flatten())
        
        all_values = np.concatenate(all_diffs)
        if vmin is None:
            vmin = np.nanmin(all_values)
        if vmax is None:
            vmax = np.nanmax(all_values)
    
    # Plotar cada comparação
    for idx, (df_orig, df_imp, label) in enumerate(df_pairs):
        row = idx // n_cols
        col = idx % n_cols
        ax = axes[row, col]
        
        # Remover NaNs do original
        df_orig_clean = df_orig.dropna(axis=1, how='all').dropna(axis=0, how='all')
        
        # Alinhar colunas entre original e imputado
        common_cols = df_orig_clean.columns.intersection(df_imp.columns)
        
        if len(common_cols) == 0:
            ax.text(0.5, 0.5, 'Sem colunas em comum', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(label, fontsize=14, fontweight='bold')
            continue
        
        # Calcular diferença nas correlações
        corr_orig = df_orig_clean[common_cols].corr()
        corr_imp = df_imp[common_cols].corr()
        corr_diff = abs(corr_orig - corr_imp)
        
        # Criar heatmap
        sns.heatmap(
            corr_diff,
            ax=ax,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            annot=annot,
            fmt=".3f",
            cbar=cbar,
            square=True,
            linewidths=0.5,
            cbar_kws={'label': 'Diferença Absoluta'} if cbar else None,
            xticklabels=show_tick_labels,
            yticklabels=show_tick_labels
        )
        
        # Configurar título
        ax.set_title(label, fontsize=14, fontweight='bold', pad=10)
        
        # Configurar labels dos eixos
        if show_axis_labels:
            ax.set_xlabel('Features', fontsize=10)
            ax.set_ylabel('Features', fontsize=10)
        else:
            ax.set_xlabel('')
            ax.set_ylabel('')
        
        # Rotacionar labels se estiverem visíveis
        if show_tick_labels:
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
            ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
        
        # Adicionar estatísticas no canto
        mean_diff = corr_diff.values[np.triu_indices_from(corr_diff.values, k=1)].mean()
        max_diff = corr_diff.values[np.triu_indices_from(corr_diff.values, k=1)].max()
        
        textstr = f'Média: {mean_diff:.4f}\nMáx: {max_diff:.4f}'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=9,
               verticalalignment='top', bbox=props)
    
    # Remover subplots vazios
    for idx in range(n_comparisons, n_rows * n_cols):
        row = idx // n_cols
        col = idx % n_cols
        fig.delaxes(axes[row, col])
    
    # Título geral
    fig.suptitle(
        'Comparação de Estruturas de Correlação: Original vs Imputado',
        fontsize=18,
        fontweight='bold',
        y=0.995
    )
    
    plt.tight_layout()

In [ ]:
from sklearn.preprocessing import StandardScaler

def compare_distribution(df_orig, df_imp, column_name):
    """
    Compara Distribuição de Valores entre Original (sem NaNs) vs Imputado (Completo)
    Usando o teste de Kolmogorov-Smirnov
    """
    ref_column = df_orig[column_name]
    imp_column = df_imp[column_name]
        
    # Filter to only include rows where ref_column is NaN
    mask = ref_column.isna()

    values_original = ref_column[~mask]
    values_imputed = imp_column[mask]

    scaler = StandardScaler()
    values_original = scaler.fit_transform(values_original.values.reshape(-1, 1)).flatten()
    values_imputed = scaler.transform(values_imputed.values.reshape(-1, 1)).flatten()

    # Contínuo: Comparar Distribuições com Wasserstein
    wass_distance = stats.wasserstein_distance(values_original, values_imputed)
    return wass_distance, len(values_imputed)

# Check Structure Differences

In [ ]:
aux = df_covid_raw.isna().sum()
aux = aux[aux > 0]
names = aux.index.tolist()

In [ ]:
pairs = [
    (df_covid_raw[names], data_covid[names], "COVID - MICE Imputation"),
    (df_covid_raw[names], data_covid_pmm[names], "COVID - PMM Imputation"),
    (df_covid_raw[names], data_covid_mixed[names], "COVID - Mixed Imputation"),
    (df_control_pre_raw[names], data_control_pre[names], "Control Pre - MICE Imputation"),
    (df_control_pre_raw[names], data_control_pre_pmm[names], "Control Pre - PMM Imputation"),
    (df_control_pre_raw[names], data_control_pre_mixed[names], "Control Pre - Mixed Imputation"),
    (df_control_post_raw[names], data_control_post[names], "Control Post - MICE Imputation"),
    (df_control_post_raw[names], data_control_post_pmm[names], "Control Post - PMM Imputation"),
    (df_control_post_raw[names], data_control_post_mixed[names], "Control Post - Mixed Imputation")
]

In [ ]:
compare_structure_multiple(pairs)

In [ ]:
compare_structure(df_control_pre_raw[names], data_control_pre_pmm[names])

# Numeric Data Comparison

### COVID

In [ ]:
data_covid = data_covid.drop(columns=['subject_id', 'hadm_id'])
data_size = data_covid.shape[0]

In [ ]:
mice = []
pmm = []
mixed = []

for col in names:

    if data_covid[col].nunique() > 2:

        mice.append(compare_distribution(df_covid_raw, data_covid, col))
        pmm.append(compare_distribution(df_covid_raw, data_covid_pmm, col))
        mixed.append(compare_distribution(df_covid_raw, data_covid_mixed, col))

print(f"MICE  - Average Difference in Numeric Columns: {np.mean([d[0]*d[1]/data_size for d in mice]):.4f}")
print(f"PMM   - Average Difference in Numeric Columns: {np.mean([d[0]*d[1]/data_size for d in pmm]):.4f}")
print(f"Mixed - Average Difference in Numeric Columns: {np.mean([d[0]*d[1]/data_size for d in mixed]):.4f}")


### Pre-Control

In [ ]:
data_control_pre = data_control_pre.drop(columns=['subject_id', 'hadm_id'])
data_size = data_control_pre.shape[0]

In [ ]:
mice = []
pmm = []
mixed = []

for col in names:

    if data_control_pre[col].nunique() > 2:

        mice.append(compare_distribution(df_control_pre_raw, data_control_pre, col))
        pmm.append(compare_distribution(df_control_pre_raw, data_control_pre_pmm, col))
        mixed.append(compare_distribution(df_control_pre_raw, data_control_pre_mixed, col))
        
print(f"MICE - Average Difference in Numeric Columns: {np.mean([d[0]*d[1]/data_size for d in mice]):.4f}")
print(f"PMM  - Average Difference in Numeric Columns: {np.mean([d[0]*d[1]/data_size for d in pmm]):.4f}")
print(f"Mixed- Average Difference in Numeric Columns: {np.mean([d[0]*d[1]/data_size for d in mixed]):.4f}")
